# 노트북 5. 생성 파라미터

> Phase 2 — 제어: 모델 동작을 원하는 대로 다루기

같은 모델, 같은 프롬프트인데 결과가 매번 다릅니다.
이유를 이해하고, 용도에 맞게 제어할 수 있어야 합니다.

**학습 목표**
- Temperature가 토큰 선택 확률 분포에 미치는 영향을 이해한다
- Top-p, Top-k의 역할과 Temperature와의 상호작용을 설명할 수 있다
- max_output_tokens, stop_sequences, seed를 활용하여 출력을 제어할 수 있다
- 용도별 최적 파라미터 조합을 스스로 도출할 수 있다
- LangChain에서 동일한 파라미터를 적용하는 방법을 안다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | Temperature + Top-p + Top-k + 기타 파라미터 + LangChain 연동 | 읽고 실행 |
| Part 2 — 실습 | 파라미터별 실험 + 조합 매트릭스 + 실전 적용 | 직접 작성 |
| Part 3 — 챌린지 | 용도별 최적 파라미터 가이드 도출 | 강사와 함께 |

In [1]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"
print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.5 MB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 왜 같은 질문에 다른 결과가 나오는가?

LLM은 다음 토큰을 **확률적으로** 선택합니다.
"오늘 날씨가" 다음에 올 수 있는 토큰이 여러 개 있고, 모델은 그 중 하나를 확률에 따라 "뽑습니다".

```
"오늘 날씨가" → 좋습니다(40%)  맑습니다(25%)  흐립니다(15%)  ...
```

이 확률 분포를 어떻게 조절하느냐에 따라 모델의 출력이 달라집니다.
이것을 조절하는 도구가 **생성 파라미터(Generation Parameters)**입니다.

| 파라미터 | 역할 | 주요 값 범위 |
|---------|------|------------|
| temperature | 확률 분포의 뾰족함 | 0.0 ~ 2.0 |
| top_p | 누적 확률 기반 필터링 | 0.0 ~ 1.0 |
| top_k | 상위 k개 후보만 남김 | 1 ~ 100+ |
| max_output_tokens | 출력 길이 상한 | 1 ~ 65,536 |
| stop_sequences | 생성 중단 트리거 | 문자열 리스트 |
| seed | 재현성을 위한 시드 | 정수 |

## 1.2 Temperature

**Temperature**(온도 파라미터)는 토큰 선택 확률 분포의 "뾰족함"을 조절합니다.

- **temperature = 0**: 가장 확률이 높은 토큰만 선택 (거의 결정적)
- **temperature = 1.0**: 원래 학습된 확률 분포 그대로
- **temperature > 1.0**: 분포가 평탄해져 낮은 확률의 토큰도 선택될 수 있음

> 핵심: temperature는 "창의성 다이얼"이라고 생각하면 됩니다.
> 낮으면 정확하고 일관되지만 뻔한 답, 높으면 다양하지만 엉뚱한 답.

아래 코드는 temperature 0.0과 1.5에서 동일한 프롬프트의 결과가 어떻게 달라지는지 보여줍니다.

In [2]:
# temperature에 따른 응답 차이 비교
prompt = "인공지능의 미래를 한 문장으로 예측해줘"

# temperature=0.0: 거의 결정적 — 매번 비슷한 답
response_low = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
)
print(f"[temperature=0.0]\n{response_low.text}")

print("---")

# temperature=1.5: 높은 무작위성 — 매번 다른 답
response_high = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"temperature": 1.5, "thinking_config": {"thinking_budget": 0}},
)
print(f"[temperature=1.5]\n{response_high.text}")

[temperature=0.0]
인공지능은 인간의 지능을 모방하고 확장하며, 예측 불가능한 방식으로 사회 전반을 재편할 것입니다.
---
[temperature=1.5]
인공지능은 점진적으로 인간 지능과 감성을 모방, 강화하며 사회 전반에 깊이 스며들어 인류의 삶을 근본적으로 재편할 것이다.


### temperature = 0의 결정성

temperature를 0으로 설정하면 항상 가장 확률 높은 토큰을 선택합니다.
이론적으로는 매번 동일한 결과가 나와야 하지만, 실제로는 서버 내부의 병렬 처리 등으로 미세한 차이가 발생할 수 있습니다.

아래 코드는 temperature=0으로 같은 질문을 3번 호출하여 결과를 비교합니다.

In [3]:
# temperature=0 반복 호출 — 결정성 확인
prompt = "대한민국의 수도는?"

results = []
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
    )
    results.append(resp.text.strip())
    print(f"호출 {i+1}: {results[-1]}")

# 결과 일치 확인
all_same = len(set(results)) == 1
print(f"\n모든 결과 동일: {all_same}")

호출 1: 대한민국의 수도는 **서울**입니다.
호출 2: 대한민국의 수도는 **서울**입니다.
호출 3: 대한민국의 수도는 **서울**입니다.

모든 결과 동일: True


### Temperature 스케일 비교

temperature를 세밀하게 조절하면 다양성을 미세 조정할 수 있습니다.
일반적인 사용 범위는 다음과 같습니다.

| temperature | 용도 | 특징 |
|------------|------|------|
| 0.0 | 분류, 추출, 코드 생성 | 가장 확률 높은 답만 선택 |
| 0.3 | 요약, Q&A | 약간의 변화 허용 |
| 0.7 | 일반 대화 | 자연스러운 다양성 |
| 1.0 | 창작, 브레인스토밍 | 학습된 분포 그대로 |
| 1.5+ | 실험적 | 매우 무작위적, 비문 가능 |

아래 코드는 5단계의 temperature에서 동일 프롬프트의 결과를 비교합니다.

In [4]:
# 5단계 temperature 비교
prompt = "커피를 마시면 좋은 점을 한 문장으로 알려줘"
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]

for temp in temperatures:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temp={temp}] {resp.text.strip()[:100]}")

[temp=0.0] 커피를 마시면 집중력 향상, 피로 감소, 기분 전환 등 다양한 긍정적인 효과를 얻을 수 있습니다.
[temp=0.3] 커피를 마시면 집중력 향상, 피로 감소, 항산화 작용 등 다양한 긍정적인 효과를 얻을 수 있습니다.
[temp=0.7] 커피는 정신을 맑게 하고 집중력을 높여 일상 활력에 도움을 줄 수 있습니다.
[temp=1.0] 커피는 정신을 맑게 하고 집중력을 높여주며, 다양한 항산화 물질로 건강에도 긍정적인 영향을 줍니다.
[temp=1.5] 커피를 마시면 집중력 향상, 운동 능력 증진, 그리고 암 및 당뇨병과 같은 질병의 위험을 줄이는 등 여러 건강상의 이점을 얻을 수 있습니다.


### 높은 Temperature와 창의적 작업

창의적 작업(소설, 시, 아이디어 발상)에서는 높은 temperature가 유용합니다.
하지만 너무 높으면 문법이 무너지거나 의미 없는 텍스트가 생성될 수 있습니다.

아래 코드는 창의적 프롬프트에서 temperature의 효과를 보여줍니다.

In [5]:
# 창의적 프롬프트에서 temperature 효과
creative_prompt = "'빗소리'라는 단어로 시작하는 짧은 시를 써줘. 2줄로."

for temp in [0.0, 0.7, 1.5]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=creative_prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"--- temperature={temp} ---")
    print(resp.text.strip())
    print()

--- temperature=0.0 ---
빗소리, 창밖을 두드리는 리듬.
내 마음도 촉촉이 젖어드네.

--- temperature=0.7 ---
빗소리, 창문을 두드려
세상은 잠시 멈춰 선다

--- temperature=1.5 ---
빗소리 창가에 부딪혀,
어둠 속 잠 못 이루는 밤.



> 정리: temperature는 가장 자주 조절하는 파라미터입니다.
> 정확성이 중요한 작업에는 0~0.3, 자연스러운 대화에는 0.5~0.7, 창의적 작업에는 0.8~1.2가 일반적입니다.

## 1.3 Top-p (Nucleus Sampling)

**Top-p**(또는 Nucleus Sampling)는 누적 확률이 p에 도달할 때까지의 토큰만 후보로 남기는 필터입니다.

예시: `top_p = 0.9`일 때
```
좋습니다(40%) + 맑습니다(25%) + 흐립니다(15%) + 따뜻합니다(10%) = 90%
→ 이 4개만 후보로 남기고, 나머지 토큰은 제외
```

- **top_p = 1.0**: 필터링 없음 (기본값, 모든 토큰이 후보)
- **top_p = 0.9**: 상위 90% 확률의 토큰만 후보
- **top_p = 0.5**: 상위 50% 확률의 토큰만 후보 (매우 제한적)

> 핵심: Top-p는 "확률이 너무 낮은 엉뚱한 토큰을 걸러내는 안전망"입니다.

아래 코드는 top_p 값에 따른 응답 차이를 비교합니다.

In [6]:
# top_p에 따른 응답 비교
prompt = "봄에 하면 좋은 활동 3가지를 추천해줘"
top_p_values = [1.0, 0.9, 0.5, 0.1]

for top_p in top_p_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,  # temperature를 고정하고 top_p만 변경
            "top_p": top_p,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    # 첫 줄만 출력
    first_line = resp.text.strip().split("\n")[0][:80]
    print(f"[top_p={top_p}] {first_line}")

[top_p=1.0] 봄에 하면 좋은 활동 3가지를 추천해 드릴게요! 🌸
[top_p=0.9] 봄에 하면 좋은 활동 3가지를 추천해 드릴게요! 🌸
[top_p=0.5] 따뜻한 봄날, 활기찬 기운을 만끽하며 즐길 수 있는 활동 3가지를 추천해 드릴게요!
[top_p=0.1] 따뜻한 봄날, 몸과 마음을 충전할 수 있는 활동 3가지를 추천해 드릴게요!


### Temperature + Top-p 상호작용

Temperature와 Top-p는 함께 작용합니다.
Temperature가 확률 분포를 변형한 뒤, Top-p가 후보를 필터링합니다.

```
원래 분포 → Temperature 적용 → Top-p 필터링 → 최종 토큰 선택
```

둘 다 높이면 매우 무작위적이고, 둘 다 낮추면 매우 결정적입니다.

| 조합 | 특징 |
|------|------|
| temp=0, top_p=1.0 | 완전 결정적 |
| temp=0.7, top_p=0.9 | 일반적 추천 조합 |
| temp=1.5, top_p=0.95 | 높은 다양성, 안전망 있음 |
| temp=1.5, top_p=1.0 | 최대 무작위 (비문 위험) |

아래 코드는 4가지 조합의 결과를 비교합니다.

In [10]:
# Temperature + Top-p 조합 비교
prompt = "AI가 미래에 대체할 직업은?"

combos = [
    (0.0, 1.0, "결정적"),
    (0.7, 0.9, "일반 추천"),
    (1.5, 0.95, "높은 다양성"),
    (1.5, 1.0, "최대 무작위"),
]

for _ in range(3):
    for temp, top_p, label in combos:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": temp,
                "top_p": top_p,
                "max_output_tokens": 60,
                "thinking_config": {"thinking_budget": 0},
            },
        )
        text = resp.text.strip().replace("\n", " ")[:80]
        print(f"[{label}: temp={temp}, top_p={top_p}]")
        print(f"  {text}\n")
    print("="*50)

[결정적: temp=0.0, top_p=1.0]
  AI 기술이 발전하면서 미래에 AI가 대체할 가능성이 높은 직업들은 다음과 같습니다.  **1. 반복적이고 예측 가능한 업무:**  *   **

[일반 추천: temp=0.7, top_p=0.9]
  AI가 미래에 대체할 가능성이 높은 직업들은 다음과 같은 특징을 가집니다:  *   **반복적이고 규칙적인 업무:** 예측 가능하고 반복적인 작

[높은 다양성: temp=1.5, top_p=0.95]
  AI가 미래에 대체할 직업은 크게 **반복적이고 예측 가능한 업무**를 수행하는 직업들일 가능성이 높습니다. 데이터를 처리하고 분석하며, 규칙 

[최대 무작위: temp=1.5, top_p=1.0]
  AI 기술이 지속적으로 발전함에 따라 다양한 직업에 영향을 미칠 것으로 예상됩니다. 특히 반복적이고 예측 가능하며, 데이터 분석에 기반한 업무들

[결정적: temp=0.0, top_p=1.0]
  AI 기술이 발전하면서 미래에 AI가 대체할 가능성이 높은 직업들은 다음과 같습니다.  **1. 반복적이고 예측 가능한 업무:**  *   **

[일반 추천: temp=0.7, top_p=0.9]
  AI가 미래에 대체할 가능성이 높은 직업은 반복적이고 예측 가능한 업무를 수행하는 직업들입니다. 특히 다음과 같은 특징을 가진 직업들이 AI 대

[높은 다양성: temp=1.5, top_p=0.95]
  인공지능(AI) 기술의 발달로 미래에 많은 직업들이 대체될 것이라는 전망이 있습니다. 특히 반복적이고 예측 가능한 작업을 수행하는 직업들이 큰 

[최대 무작위: temp=1.5, top_p=1.0]
  AI 기술이 빠르게 발전하면서 미래에는 지금과 다른 형태의 직업들이 생겨날 것으로 예상됩니다. 특히, 인간의 **창의성, 공감 능력, 전략적 사

[결정적: temp=0.0, top_p=1.0]
  AI 기술이 발전하면서 미래에 AI가 대체할 가능성이 높은 직업들은 다음과 같습니다.  **1. 반복적이고 예측 가능

> 실무 가이드라인: 대부분의 경우 temperature만 조절하면 충분합니다.
> top_p는 0.9~0.95로 고정하고, temperature로 다양성을 조절하는 것이 가장 일반적인 패턴입니다.
> 둘 다 극단적으로 설정하는 것은 피하세요.

## 1.4 Top-k

**Top-k**는 확률 상위 k개 토큰만 후보로 남기는 필터입니다.
Top-p가 확률 비율로 필터링하는 반면, Top-k는 개수로 필터링합니다.

- **top_k = 1**: 가장 확률 높은 토큰 1개만 (= temperature 0과 유사)
- **top_k = 40**: 상위 40개 토큰 중에서 선택 (Gemini 기본값)
- **top_k = 100+**: 거의 필터링 없음

> 참고: Top-k는 Gemini에서는 지원하지만 OpenAI API에는 없는 파라미터입니다.
> 모델마다 지원하는 파라미터가 다르므로, 문서를 확인하는 습관이 중요합니다.

아래 코드는 top_k 값에 따른 응답 차이를 비교합니다.

In [11]:
# top_k에 따른 응답 비교
prompt = "재미있는 파이썬 프로젝트 아이디어를 하나 제안해줘"
top_k_values = [1, 10, 40, 100]

for top_k in top_k_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "top_k": top_k,
            "max_output_tokens": 60,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    text = resp.text.strip().split("\n")[0][:80]
    print(f"[top_k={top_k:>3}] {text}")

[top_k=  1] ## 재미있는 파이썬 프로젝트 아이디어: AI 기반의 "나만의 음악 작곡가"
[top_k= 10] 재미있는 파이썬 프로젝트 아이디어 하나 제안해 드릴게요! 🤖✨
[top_k= 40] 좋습니다! 파이썬의 강력함과 다양한 라이브러리를 활용하면서 재미있게 배울 수 있는 프로젝트 아이디어를 하나 제안해 드릴게요.
[top_k=100] ## 재미있는 파이썬 프로젝트 아이디어: AI 기반 추천 시스템 만들기 (영화, 책 등)


> 참고: Top-k와 Top-p의 차이
> - Top-k: 항상 정확히 k개의 후보. 확률 분포의 모양과 무관
> - Top-p: 확률 분포에 따라 후보 수가 동적으로 변함. 분포가 뾰족하면 적은 수, 평탄하면 많은 수
> - 실무에서는 Top-p가 더 많이 사용됩니다. 분포 적응적이기 때문입니다.

## 1.5 파라미터 조합 전략

Temperature, Top-p, Top-k는 독립적으로 사용할 수도 있지만, 함께 조합하는 것이 일반적입니다.
아래는 용도별 추천 프리셋입니다.

| 용도 | temperature | top_p | top_k | 설명 |
|------|-----------|-------|-------|------|
| 코드 생성 | 0.0 | 1.0 | - | 정확성 최우선 |
| 분류/추출 | 0.0~0.2 | 1.0 | - | 일관된 결과 |
| 요약/Q&A | 0.3 | 0.95 | 40 | 약간의 변화 허용 |
| 일반 대화 | 0.7 | 0.9 | 40 | 자연스러운 다양성 |
| 창작/브레인스토밍 | 1.0~1.2 | 0.95 | 60 | 높은 다양성 |

아래 코드는 프리셋별 결과를 비교합니다.

In [12]:
# 용도별 프리셋 비교
presets = {
    "코드 생성": {"temperature": 0.0, "top_p": 1.0},
    "요약/Q&A": {"temperature": 0.3, "top_p": 0.95, "top_k": 40},
    "일반 대화": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "창작": {"temperature": 1.0, "top_p": 0.95, "top_k": 60},
}

prompt = "'시간'이라는 주제로 한 문장을 써줘"

for name, params in presets.items():
    config = {**params, "thinking_config": {"thinking_budget": 0}}
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config,
    )
    print(f"[{name}] {resp.text.strip()[:100]}")

[코드 생성] 시간은 덧없이 흐르지만, 그 속에서 우리는 영원을 꿈꾼다.
[요약/Q&A] 시간은 덧없이 흐르지만, 그 속에서 우리는 영원한 의미를 찾는다.
[일반 대화] '시간은 멈추지 않고 흐르며, 모든 순간은 다시 오지 않을 소중한 선물이다.'
[창작] 시간은 덧없이 흐르지만, 그 흐름 속에서 우리는 영원을 꿈꾼다.


### 매트릭스 실험

Temperature와 Top-p의 조합을 체계적으로 실험해봅시다.
표 형태로 결과를 정리하면 패턴을 쉽게 파악할 수 있습니다.

아래 코드는 3x3 매트릭스 실험을 수행합니다.

매트릭스 결과를 관찰하면 몇 가지 패턴이 보입니다.

- **temp=0.0 행**: top_p를 바꿔도 결과가 거의 동일합니다. temperature가 0이면 이미 하나의 토큰만 선택하므로 top_p 필터링이 무의미합니다.
- **temp=1.5 행**: top_p에 따라 결과가 크게 달라집니다. 높은 temperature로 분포가 평탄해진 상태에서 top_p가 안전망 역할을 합니다.
- **대각선**: temp=0/top_p=1.0(가장 결정적) → temp=1.5/top_p=1.0(가장 무작위)으로 갈수록 다양성이 증가합니다.

In [15]:
# Temperature x Top-p 매트릭스 실험
prompt = "사과가 빨간 이유를 한 문장으로 설명해줘"
temps = [0.0, 0.8, 2.0]
top_ps = [0.3, 0.6, 1.0]

print(f"{'':>12}", end="")
for tp in top_ps:
    print(f"{'top_p='+str(tp):>30}", end="")
print()
print("-" * 102)

for temp in temps:
    print(f"temp={temp:<5}", end="  ")
    for tp in top_ps:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": temp,
                "top_p": tp,
                "max_output_tokens": 30,
                "thinking_config": {"thinking_budget": 0},
            },
        )
        text = resp.text.strip().replace("\n", " ")[:25]
        print(f"{text:>30}", end="")
    print()

                                 top_p=0.3                     top_p=0.6                     top_p=1.0
------------------------------------------------------------------------------------------------------
temp=0.0         사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과가 빨간 이유는 안토시아닌이라는 색소 때문
temp=0.8         사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과가 빨간 이유는 안토시아닌이라는 색소 때문
temp=2.0         사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과가 빨간 이유는 안토시아닌이라는 색소 때문     사과는 완숙되면 맛있는 물질이 많아져 새빨갛게


## 1.6 max_output_tokens

**max_output_tokens**는 모델이 생성하는 출력의 최대 토큰 수를 제한합니다.

- 설정하지 않으면 모델 기본 최대값(gemini-2.5-flash: 65,536) 적용
- 짧은 답변이 필요할 때 비용과 속도를 절약할 수 있음
- 토큰 한도에 도달하면 문장 중간에서 잘릴 수 있음

아래 코드는 max_output_tokens에 따른 출력 길이 차이를 보여줍니다.

In [16]:
# max_output_tokens에 따른 출력 길이
prompt = "파이썬 프로그래밍의 장점을 설명해주세요."
max_tokens_list = [20, 50, 200]

for max_tokens in max_tokens_list:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "max_output_tokens": max_tokens,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    usage = resp.usage_metadata
    print(f"[max={max_tokens:>3}] 실제 출력: {usage.candidates_token_count}토큰")
    print(f"  {resp.text.strip()[:120]}")
    print()

[max= 20] 실제 출력: 20토큰
  파이썬 프로그래밍은 다양한 분야에서 폭넓게 사용되며, 여러 가지 강력한

[max= 50] 실제 출력: 50토큰
  파이썬 프로그래밍은 다양한 분야에서 널리 사용되며, 다음과 같은 많은 장점을 가지고 있습니다.

**1. 배우기 쉽고 직관적입니다.**

* **간결한 문법:** 파이썬은

[max=200] 실제 출력: 200토큰
  파이썬 프로그래밍은 다양한 장점을 가지고 있어 많은 개발자와 기업에서 선호하고 있습니다. 주요 장점들을 자세히 설명해 드릴게요.

## 파이썬 프로그래밍의 장점

### 1. 배우기 쉽고 직관적인 문법 (Easy t



### finish_reason 확인

max_output_tokens에 도달하여 생성이 중단되면, 응답의 `finish_reason`이 `MAX_TOKENS`로 표시됩니다.
정상적으로 완료된 경우에는 `STOP`입니다.

이 정보를 활용하면 "응답이 잘렸는지" 프로그래밍적으로 판단할 수 있습니다.

아래 코드는 finish_reason을 확인합니다.

In [17]:
# finish_reason 확인
prompt = "1부터 100까지의 소수를 모두 나열해줘"

# 충분한 토큰 → 정상 완료
resp_full = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 500, "thinking_config": {"thinking_budget": 0}},
)
print(f"[max=500] finish_reason: {resp_full.candidates[0].finish_reason}")
print(f"  출력 토큰: {resp_full.usage_metadata.candidates_token_count}")

print()

# 부족한 토큰 → 중간 절단
resp_short = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 20, "thinking_config": {"thinking_budget": 0}},
)
print(f"[max=20]  finish_reason: {resp_short.candidates[0].finish_reason}")
print(f"  출력 토큰: {resp_short.usage_metadata.candidates_token_count}")
print(f"  잘린 텍스트: {resp_short.text.strip()[:80]}...")

[max=500] finish_reason: FinishReason.STOP
  출력 토큰: 108

[max=20]  finish_reason: FinishReason.MAX_TOKENS
  출력 토큰: 19
  잘린 텍스트: 1부터 100까지의 소수는 다음과 같습니다:

2, 3,...


> 실무 팁: max_output_tokens는 비용 제어 수단으로도 유용합니다.
> 답변이 길어질 필요가 없는 분류/추출 작업에서는 50~100으로 제한하면 출력 토큰 비용을 절약할 수 있습니다.
> 단, 잘린 응답(finish_reason=MAX_TOKENS)이 사용자에게 노출되지 않도록 후처리가 필요합니다.

## 1.7 stop_sequences

**stop_sequences**는 모델이 특정 문자열을 생성하면 즉시 생성을 중단하는 파라미터입니다.
출력 포맷을 제어하거나, 불필요한 부분의 생성을 방지할 때 유용합니다.

- 최대 5개의 중단 시퀀스를 지정할 수 있음
- 중단 문자열 자체는 출력에 포함되지 않음

아래 코드는 stop_sequences로 생성을 중단하는 예시를 보여줍니다.

In [18]:
# stop_sequences 기본 사용
prompt = "1부터 10까지 세어줘: 1, 2, 3, "

# stop_sequences 없이
resp_no_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)
print(f"[stop 없음] {resp_no_stop.text.strip()[:60]}")

# "7"에서 중단
resp_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["7"],
        "thinking_config": {"thinking_budget": 0},
    },
)
print(f"[stop='7'] {resp_stop.text.strip()[:60]}")

[stop 없음] 1부터 10까지 세어줄게: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10.
[stop='7'] 1부터 10까지 세어줄게: 1, 2, 3, 4, 5, 6,


### 복수 stop_sequences와 실용적 활용

여러 개의 중단 시퀀스를 동시에 지정할 수 있습니다.
실용적인 활용 예시:

- 구분자 기반 파싱: `"\n\n"`으로 첫 단락만 가져오기
- 리스트 제한: `"4."` 으로 상위 3개만 가져오기
- 코드 블록 끝: `"```"` 으로 코드 블록만 추출

아래 코드는 복수 stop_sequences를 활용하는 예시입니다.

In [19]:
# 복수 stop_sequences 활용
# 리스트에서 상위 3개만 가져오기
prompt = "좋은 프로그래밍 습관 5가지를 번호를 매겨 알려줘"

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["4.", "4)"],  # 4번 항목이 시작되면 중단
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[상위 3개만 출력]")
print(resp.text.strip())

print("\n" + "=" * 40)

# 첫 문장만 가져오기
prompt2 = "딥러닝에 대해 설명해줘"
resp2 = client.models.generate_content(
    model=MODEL,
    contents=prompt2,
    config={
        "stop_sequences": ["\n"],  # 첫 줄바꿈에서 중단
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[첫 문장만]")
print(resp2.text.strip())

[상위 3개만 출력]
좋은 프로그래밍 습관 5가지를 번호 매겨 알려드릴게요.

1.  **계획 및 설계:** 코드를 작성하기 전에 무엇을 만들고 어떻게 작동할지 충분히 생각하고 설계하는 습관입니다. 예를 들어, 간단한 의사코드(pseudocode)를 작성하거나, 클래스 다이어그램을 그리거나, 주요 함수들의 역할을 미리 정의해보는 것입니다. 이렇게 하면 전체적인 그림을 파악하고 예상치 못한 문제를 줄일 수 있습니다.

2.  **테스트 주도 개발 (TDD) 또는 잦은 테스트:** 코드를 작성하기 전 또는 작성 직후 해당 기능이 제대로 작동하는지 확인하는 테스트 코드를 작성하거나, 적어도 개발 과정에서 자주 기능을 테스트해보는 습관입니다. 이는 버그를 조기에 발견하고, 코드가 의도한 대로 동작하는지 지속적으로 검증하며, 나중에 코드를 변경할 때도 안전하게 리팩토링할 수 있도록 돕습니다.

3.  **코드 리뷰 참여 및 요청:** 동료 개발자에게 자신의 코드를 검토해달라고 요청하고, 다른 사람의 코드를 검토해주는 습관입니다. 코드 리뷰를 통해 다양한 관점에서 코드의 개선점을 찾을 수 있고, 잠재적인 버그를 미리 발견하며, 팀 전체의 코드 품질을 향상시킬 수 있습니다. 또한, 다른 사람의 코드를 보면서 새로운 아이디어나 더 나은 구현 방법을 배울 수 있습니다.

[첫 문장만]
## 딥러닝 (Deep Learning) 이란?


## 1.8 seed

**seed**는 난수 생성의 시작점을 고정하여 재현 가능한 결과를 만드는 파라미터입니다.
같은 seed, 같은 프롬프트, 같은 파라미터를 사용하면 동일한 결과가 나올 가능성이 높아집니다.

- 실험 재현, A/B 테스트, 디버깅에 유용
- temperature > 0일 때도 결과를 고정할 수 있음

아래 코드는 seed를 사용한 재현성을 확인합니다.

In [20]:
# seed를 사용한 재현성 확인
prompt = "재미있는 동물 이름을 하나 지어줘"

# seed 없이 (매번 다름)
print("=== seed 없이 (temperature=1.0) ===")
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 1.0, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"  호출 {i+1}: {resp.text.strip()[:50]}")

print()

# seed 고정 (재현 가능)
print("=== seed=42 (temperature=1.0) ===")
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "seed": 42,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    print(f"  호출 {i+1}: {resp.text.strip()[:50]}")

=== seed 없이 (temperature=1.0) ===
  호출 1: 재미있는 동물 이름이요! 음... 어떤 동물을 위한 이름이냐에 따라 다르겠지만, 일반적인 
  호출 2: 물론이죠! 재미있는 동물 이름을 지어볼까요?

**"반짝이 발가락 대마왕"**

어떤가요?
  호출 3: 음, 어떤 동물이 떠오르시나요?

**재미있는 동물 이름 지어드릴게요!**

* **수식어

=== seed=42 (temperature=1.0) ===
  호출 1: 재밌는 동물 이름 지어드릴게요! 어떤 동물이 될까요? 혹시 생각하고 있는 동물이 있나요?

  호출 2: 재밌는 동물 이름 지어드릴게요! 어떤 동물이 될까요? 혹시 생각하고 있는 동물이 있나요?

  호출 3: 재밌는 동물 이름 지어드릴게요! 어떤 동물이 될까요? 혹시 생각하고 있는 동물이 있나요?



### seed의 한계

seed를 사용해도 100% 동일한 결과가 보장되지는 않습니다.

- 모델 업데이트 시 결과가 달라질 수 있음
- 서버 인프라 변경에 영향을 받을 수 있음
- 같은 seed라도 프롬프트가 다르면 당연히 결과도 다름

> 핵심: seed는 "최선의 노력(best-effort)" 재현성입니다.
> 완벽한 재현이 필요하면 응답을 캐싱하는 것이 더 확실합니다.

아래 코드는 seed를 다르게 설정했을 때의 결과를 비교합니다.

> 파라미터 적용 우선순위 정리
> 1. **temperature**: 가장 먼저 조절. 용도에 따라 0.0~1.2 범위에서 결정
> 2. **top_p**: 기본값 유지(0.9~0.95)가 대부분 충분. 극단적 다양성이 필요할 때만 조절
> 3. **max_output_tokens**: 비용 제어와 응답 길이 제한에 사용
> 4. **stop_sequences**: 특정 포맷 제어가 필요할 때만 사용
> 5. **seed**: 실험 재현이 필요할 때만 설정
> 6. **top_k**: Gemini 전용. 일반적으로 기본값(40)을 유지

In [21]:
# 서로 다른 seed → 서로 다른 결과
prompt = "오늘의 명언을 하나 만들어줘"

for seed_val in [1, 42, 100, 999]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "seed": seed_val,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    print(f"[seed={seed_val:>3}] {resp.text.strip()[:80]}")

[seed=  1] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"꿈을 향한 작은 한 걸음이, 결국 당신의 세상을 바꿀 지도가 된다."**

이 명언이 당신에게
[seed= 42] 네, 오늘의 명언을 하나 만들어 드릴게요.

---

**오늘의 명언:**

**"어제의 후회에 갇히지 말고, 내일의 불안에 흔들리지 마라. 오
[seed=100] ## 오늘의 명언:

**"바람에 흔들릴지언정, 뿌리 깊은 나무는 꺾이지 않는다. 흔들림 속에서 너의 단단함을 찾아라."**

---

**설명
[seed=999] 물론입니다! 오늘의 명언을 하나 만들어 드릴게요.

**"어제는 오늘의 스승이었고, 내일은 오늘이 빚어낼 찬란함이다."**

이 명언은 과거의 


## 1.9 google-genai config 총정리

지금까지 배운 모든 파라미터를 하나의 config에 모아서 사용할 수 있습니다.

아래 코드는 모든 생성 파라미터를 한 번에 설정하는 예시를 보여줍니다.

In [22]:
# 모든 생성 파라미터를 한 번에 설정
response = client.models.generate_content(
    model=MODEL,
    contents="좋은 코딩 습관 3가지를 알려줘",
    config={
        "temperature": 0.3,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 200,
        "stop_sequences": ["4.", "4)"],  # 3개까지만
        "seed": 42,
        "thinking_config": {"thinking_budget": 0},
    },
)

print(response.text.strip())
print(f"\n--- 메타데이터 ---")
print(f"출력 토큰: {response.usage_metadata.candidates_token_count}")
print(f"finish_reason: {response.candidates[0].finish_reason}")

좋은 코딩 습관은 코드를 더 읽기 쉽고, 유지보수하기 쉬우며, 오류를 줄이는 데 도움이 됩니다. 다음은 3가지 핵심 습관입니다.

---

### 1. **명확하고 일관된 코드 스타일 유지하기**

코드는 컴퓨터뿐만 아니라 다른 사람(미래의 나 자신 포함)도 읽고 이해해야 합니다.

*   **변수 및 함수 이름:** 의미를 명확하게 전달하는 이름을 사용하세요. (예: `calculateTotalPrice` 대신 `calc` 사용 지양)
*   **들여쓰기:** 일관된 들여쓰기를 사용하여 코드 블록의 구조를 명확하게 보여줍니다. (탭 또는 스페이스, 팀 규칙 따르기)
*   **공백:** 연산자 주변이나 괄호 뒤에 적절한 공백을 사용하여 가독성을 높입니다.
*   **줄 길이:** 너무 긴

--- 메타데이터 ---
출력 토큰: 200
finish_reason: FinishReason.MAX_TOKENS


## 1.10 LangChain에서 파라미터 적용

LangChain의 `ChatGoogleGenerativeAI`에서도 동일한 파라미터를 사용할 수 있습니다.
생성자에서 직접 설정하거나, 호출 시 덮어쓸 수 있습니다.

아래 코드는 LangChain에서 생성 파라미터를 설정하는 방법을 보여줍니다.

In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 생성자에서 파라미터 설정
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    top_p=0.95,
    top_k=40,
    max_output_tokens=200,
    thinking_budget=0,
)

response = llm.invoke("파이썬의 장점을 한 문장으로 알려줘")
print(f"[LangChain temp=0.3]")
print(response.content)
print(f"\ntoken_usage: {response.usage_metadata}")

[LangChain temp=0.3]
**파이썬은 배우기 쉽고 활용 분야가 넓어 빠르게 아이디어를 구현할 수 있는 강력한 언어입니다.**

token_usage: {'input_tokens': 13, 'output_tokens': 28, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}}


### LCEL 체인에서 파라미터

LCEL 체인(`prompt | model | parser`)에서도 모델의 생성 파라미터가 그대로 적용됩니다.
체인을 여러 개 만들어 파라미터 조합별로 비교할 수도 있습니다.

아래 코드는 LCEL 체인에서 서로 다른 temperature를 사용하는 예시입니다.

In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 답변하세요."),
    ("human", "{question}"),
])
parser = StrOutputParser()

# 같은 체인 구조, 다른 temperature
for temp in [0.0, 0.7, 1.2]:
    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GEMINI_API_KEY,
        temperature=temp,
    )
    chain = prompt_template | model.bind(stop=["문제"]) | parser
    result = chain.invoke({"question": "프로그래밍을 배워야 하는 이유는?"})
    print(f"[temp={temp}] {result.strip()[:100]}")

[temp=0.0] 
[temp=0.7] THINK
The user is asking for the reason to learn programming.
I need to provide a concise, single-se
[temp=1.2] THOUGHT: The user wants to know why one should learn programming, and I need to provide a single-sen


> google-genai vs LangChain 파라미터 비교
>
> | 파라미터 | google-genai (config dict) | LangChain (생성자) |
> |---------|--------------------------|-------------------|
> | temperature | `"temperature": 0.3` | `temperature=0.3` |
> | top_p | `"top_p": 0.95` | `top_p=0.95` |
> | top_k | `"top_k": 40` | `top_k=40` |
> | max_output_tokens | `"max_output_tokens": 200` | `max_output_tokens=200` |
>
> LangChain은 여러 모델 제공자를 추상화하므로, stop_sequences처럼 제공자별 차이가 있는 파라미터는 `bind()` 또는 `model_kwargs`를 통해 전달합니다.

### 호출 시 파라미터 덮어쓰기

LangChain 모델은 생성자에서 기본 파라미터를 설정하고, `invoke()` 시 `config`로 일부를 덮어쓸 수 있습니다.
같은 모델 인스턴스를 다양한 상황에서 재사용할 때 유용합니다.

아래 코드는 `bind()` 메서드로 파라미터를 동적으로 변경하는 예시입니다.

In [50]:
# bind()로 파라미터 덮어쓰기
base_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.7,
)

# 기본 설정으로 호출
resp1 = base_model.invoke("재미있는 사실 하나 알려줘")
print(f"[기본 temp=0.7] {resp1.content.strip()[:80]}")

# stop_sequences 추가 바인딩
strict_model = base_model.bind(stop=["\n"])
resp2 = strict_model.invoke("재미있는 사실 하나 알려줘")
print(f"[+stop='\\n'] {resp2.content.strip()[:80]}")

[기본 temp=0.7] 여기 재미있는 사실 하나 알려드릴게요!

**달팽이는 최대 3년까지 잠을 잘 수 있습니다.**

신기하죠?
[+stop='\n'] 여기 재미있는 사실 하나 알려드릴게요!


---

### 생각해보기

1. temperature를 0으로 설정했는데도 매번 미세하게 다른 결과가 나올 수 있을까요? 그렇다면 왜일까요?
2. 코드 생성에는 낮은 temperature, 창작에는 높은 temperature가 권장됩니다. 그렇다면 "코드를 창의적으로 리팩토링해줘"라는 요청에는 어떤 값이 적절할까요?
3. temperature와 top_p를 둘 다 최대로 올리면 어떤 일이 벌어질까요? 실제로 시도해보고 결과를 관찰해보세요.

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 5-1: Temperature 응답 비교

temperature를 0.0과 1.5로 설정하여 동일한 프롬프트의 결과를 비교하세요.

**요구사항**
1. 프롬프트: "가을에 하면 좋은 일을 한 문장으로 추천해줘"
2. temperature=0.0과 temperature=1.5로 각각 호출
3. 두 결과를 출력하여 차이를 확인

In [51]:
# TODO: temperature 0.0과 1.5의 결과를 비교하세요
prompt = "가을에 하면 좋은 일을 한 문장으로 추천해줘"

# ---------- 정답 ----------
for temp in [0.0, 1.5]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temp={temp}] {resp.text.strip()[:100]}")

print("TODO를 완성해주세요")

[temp=0.0] 가을에는 선선한 바람을 맞으며 **단풍 구경**을 하거나, 따뜻한 차 한 잔과 함께 **독서**를 즐기는 것이 좋습니다.
[temp=1.5] **가을에는 선선한 바람을 맞으며 걷기 좋은 길을 찾아 여유로운 산책을 즐겨보세요.**
TODO를 완성해주세요


### 실습 5-2: Temperature 재현성 확인

temperature=0.0으로 같은 질문을 5번 호출하고, 결과가 동일한지 확인하세요.

**요구사항**
1. 프롬프트: "1+1은?"
2. temperature=0.0으로 5번 호출
3. 결과를 리스트에 저장
4. 모든 결과가 동일한지 `set()`으로 확인하여 출력

In [52]:
# TODO: temperature=0 반복 호출 재현성 확인
prompt = "1+1은?"
results = []

# ---------- 정답 ----------
for i in range(5):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
    )
    results.append(resp.text.strip())
    print(f"호출 {i+1}: {results[-1][:50]}")

unique = len(set(results))
print(f"\n고유 결과 수: {unique}/{len(results)}")
print(f"모든 결과 동일: {unique == 1}")

print("TODO를 완성해주세요")

호출 1: 1+1은 **2**입니다.
호출 2: 1+1은 **2**입니다.
호출 3: 1+1은 **2**입니다.
호출 4: 1+1은 **2**입니다.
호출 5: 1+1은 **2**입니다.

고유 결과 수: 1/5
모든 결과 동일: True
TODO를 완성해주세요


### 실습 5-3: Top-p 실험

temperature를 고정(1.0)하고, top_p를 변화시켜 결과를 비교하세요.

**요구사항**
1. 프롬프트: "주말에 할 수 있는 취미를 하나 추천해줘"
2. temperature=1.0 고정
3. top_p를 0.1, 0.5, 0.9, 1.0으로 각각 호출
4. 결과를 비교 출력

In [53]:
# TODO: top_p를 변화시켜 결과를 비교하세요
prompt = "주말에 할 수 있는 취미를 하나 추천해줘"
top_p_values = [0.1, 0.5, 0.9, 1.0]

# ---------- 정답 ----------
for tp in top_p_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "top_p": tp,
            "max_output_tokens": 60,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    text = resp.text.strip().split("\n")[0][:80]
    print(f"[top_p={tp}] {text}")

print("TODO를 완성해주세요")

[top_p=0.1] 주말에 즐길 수 있는 취미로 **베이킹**을 추천합니다! 🥐🍰
[top_p=0.5] 주말에 즐길 수 있는 취미로 **베이킹**을 추천합니다! 🥐🍰
[top_p=0.9] 주말에 즐길 수 있는 취미로 **홈베이킹**을 추천합니다! 🥐🍰
[top_p=1.0] 주말에 즐길 수 있는 취미로 **홈 가드닝**을 추천합니다! 🌱
TODO를 완성해주세요


### 실습 5-4: Temperature x Top-p 매트릭스

Temperature와 Top-p의 조합을 체계적으로 실험하세요.

**요구사항**
1. 프롬프트: "행복이란 무엇인가를 한 문장으로 정의해줘"
2. temperature: [0.0, 0.7, 1.5]
3. top_p: [0.5, 0.9, 1.0]
4. 3x3 = 9가지 조합을 모두 실행
5. 결과를 표 형태로 출력

In [55]:
# TODO: Temperature x Top-p 매트릭스 실험
prompt = "행복이란 무엇인가를 한 문장으로 정의해줘"
temps = [0.0, 0.7, 1.5]
top_ps = [0.5, 0.9, 1.0]

# ---------- 정답 ----------
for temp in temps:
    for tp in top_ps:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": temp,
                "top_p": tp,
                "max_output_tokens": 50,
                "thinking_config": {"thinking_budget": 0},
            },
        )
        text = resp.text.strip().replace("\n", " ")
        print(f"[temp={temp}, top_p={tp}] {text}")
    print()  # 그룹 구분

print("TODO를 완성해주세요")

[temp=0.0, top_p=0.5] 행복이란 삶의 긍정적인 경험과 만족감을 느끼는 주관적인 상태이다.
[temp=0.0, top_p=0.9] 행복이란 삶의 만족과 기쁨을 느끼는 긍정적인 심리 상태이다.
[temp=0.0, top_p=1.0] 행복이란 삶의 만족과 기쁨을 느끼는 긍정적인 심리 상태이다.

[temp=0.7, top_p=0.5] 행복이란 삶의 만족과 기쁨을 느끼는 긍정적인 심리 상태이다.
[temp=0.7, top_p=0.9] 행복이란 삶의 긍정적인 경험을 통해 만족감과 충만함을 느끼는 주관적인 감정 상태이다.
[temp=0.7, top_p=1.0] 행복이란 주관적인 만족감과 긍정적인 감정의 상태를 아우르는 포괄적인 경험이다.

[temp=1.5, top_p=0.5] 행복이란 주관적인 만족감과 긍정적인 감정의 총체로, 삶의 의미와 목적을 찾아가는 과정에서 느끼는 충만함이다.
[temp=1.5, top_p=0.9] 행복은 긍정적인 감정 상태와 만족감을 포괄하는 주관적인 경험입니다.
[temp=1.5, top_p=1.0] 행복이란 삶의 긍정적인 경험과 의미 속에서 개인이 느끼는 깊은 만족감이다.

TODO를 완성해주세요


### 실습 5-5: max_output_tokens 제어

max_output_tokens를 조절하여 출력 길이를 제어하고, finish_reason을 확인하세요.

**요구사항**
1. 프롬프트: "머신러닝의 종류와 각각의 특징을 설명해주세요"
2. max_output_tokens를 10, 50, 200으로 각각 호출
3. 각 결과의 실제 출력 토큰 수와 finish_reason을 출력

In [56]:
# TODO: max_output_tokens 제어 + finish_reason 확인
prompt = "머신러닝의 종류와 각각의 특징을 설명해주세요"
max_tokens_list = [10, 50, 200]

# ---------- 정답 ----------
for max_t in max_tokens_list:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"max_output_tokens": max_t, "thinking_config": {"thinking_budget": 0}},
    )
    u = resp.usage_metadata
    fr = resp.candidates[0].finish_reason
    print(f"[max={max_t:>3}] 출력: {u.candidates_token_count}토큰 | finish: {fr}")
    print(f"  {resp.text.strip()}")
    print()

print("TODO를 완성해주세요")

[max= 10] 출력: 10토큰 | finish: FinishReason.MAX_TOKENS
  머신러닝은 크게 **지도 학습(

[max= 50] 출력: 50토큰 | finish: FinishReason.MAX_TOKENS
  머신러닝은 크게 **지도 학습(Supervised Learning)**, **비지도 학습(Unsupervised Learning)**, **강화 학습(Reinforcement Learning)**의 세 가지 주요 종류로 나눌 수 있습니다. 각각의 특징과 대표

[max=200] 출력: 199토큰 | finish: FinishReason.MAX_TOKENS
  머신러닝은 크게 **지도 학습(Supervised Learning)**, **비지도 학습(Unsupervised Learning)**, ****강화 학습(Reinforcement Learning)****의 세 가지 종류로 나눌 수 있습니다. 각각의 특징을 자세히 살펴보겠습니다.

---

### 1. 지도 학습 (Supervised Learning)

**개념:** 가장 일반적인 머신러닝 방식으로, **정답(레이블)이 있는 데이터를 학습**하여 입력 데이터와 출력 데이터 간의 관계를 모델링합니다. 모델은 학습 데이터를 통해 패턴을 학습하고, 새로운 입력 데이터에 대해 예측을 수행합니다.

**작동 방식:**
1. **입력(특징)**과 **출력(레이블)**이 쌍으로 이루어진 학습 데이터를 준비합니다.
2. 모델은 입력과 출력을 매핑하는 함수를 학습합니다.
3. 학습된 모델은 새로운 입력 데이터에 대해 이전에 본 적 없는 레이블을

TODO를 완성해주세요


### 실습 5-6: stop_sequences 활용

stop_sequences를 사용하여 원하는 지점에서 생성을 중단하세요.

**요구사항**
1. 프롬프트: "대한민국의 대도시 5개를 번호를 매겨 나열해줘"
2. stop_sequences로 3번 항목까지만 출력되게 설정
3. stop_sequences 있는 결과와 없는 결과를 비교 출력

In [57]:
# TODO: stop_sequences로 생성 중단 제어
prompt = "대한민국의 대도시 5개를 번호를 매겨 나열해줘"

# ---------- 정답 ----------
# stop 없이
resp_full = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)
print("[stop 없음]")
print(resp_full.text.strip())

print("\n" + "=" * 30)

# 4번부터 차단
resp_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["4.", "4)"],
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[stop='4.'/'4)']")
print(resp_stop.text.strip())

print("TODO를 완성해주세요")

[stop 없음]
네, 대한민국 대도시 5곳을 번호 매겨 나열해 드릴게요.

1.  **서울특별시**
2.  **부산광역시**
3.  **인천광역시**
4.  **대구광역시**
5.  **대전광역시**

[stop='4.'/'4)']
네, 대한민국 대도시 5개를 번호로 나열해 드리겠습니다.

1. **서울**
2. **부산**
3. **인천**
TODO를 완성해주세요


### 실습 5-7: LangChain에서 파라미터 적용

LangChain의 LCEL 체인에서 생성 파라미터를 적용하세요.

**요구사항**
1. ChatGoogleGenerativeAI 모델 생성 (temperature=0.3, max_output_tokens=100)
2. ChatPromptTemplate: system="한국어로 간결하게 답변하세요", human="{topic}에 대해 알려줘"
3. StrOutputParser로 체인 구성
4. topic="양자컴퓨팅"으로 호출
5. 결과 출력

In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: LangChain LCEL 체인 구성 + 파라미터 적용

# ---------- 정답 ----------
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    max_output_tokens=100,
    thinking_budget=20
)

prompt_tpl = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답변하세요"),
    ("human", "{topic}에 대해 알려줘"),
])

chain = prompt_tpl | model | StrOutputParser()
result = chain.invoke({"topic": "양자컴퓨팅"})
print(result)

print("TODO를 완성해주세요")

양자컴퓨팅은 양자역학의 원리(중첩, 얽힘)를 활용하여 정보를 처리하는 새로운 방식의 컴퓨터입니다.

기존 컴퓨터가 0과 1 중 하나만 표현하는 '비트'를 사용하는 반면, 양자컴퓨터는 0과 1을 동시에 표현할 수 있는 '큐비트'
TODO를 완성해주세요


---

### 생각해보기

1. 실습 5-4의 매트릭스에서 temp=0.0일 때 top_p를 바꿔도 결과가 같았나요? 왜 그런 걸까요?
2. stop_sequences는 출력 포맷 제어에 유용하지만, Structured Output(노트북 8)을 사용하면 더 안정적입니다. 두 방식의 장단점을 비교해보세요.
3. seed를 사용하면 실험 재현이 가능하지만 100% 보장은 안 됩니다. 완벽한 재현성이 필요한 경우 어떤 전략을 쓸 수 있을까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 5-1: 용도별 최적 파라미터 가이드 도출 (난이도: ★☆☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

2가지 서로 다른 용도에 대해 최적의 파라미터 조합을 실험적으로 도출하세요.

**용도 목록**
1. 영어-한국어 번역
2. 뉴스 기사 요약

**과제**
- 각 용도에 맞는 프롬프트를 작성
- 가장 좋은 조합을 선택하고, 그 이유를 설명
- 최종 가이드를 표 형태로 정리

**힌트**
- 정확성이 중요한 작업 vs 다양성이 중요한 작업을 먼저 구분하세요
- max_output_tokens도 용도에 맞게 조절해보세요
- 같은 조합으로 3번 호출하여 일관성도 확인하세요

In [ ]:
# 1단계: 용도별 프롬프트 정의 + 파라미터 실험
# 여기에 코드를 작성하세요

In [ ]:
# 2단계: 결과 비교 및 최적 조합 선택
# 여기에 코드를 작성하세요

In [ ]:
# 3단계: 최종 가이드 표 정리
# 여기에 코드를 작성하세요

---

### 생각해보기

1. 파라미터 가이드를 만들었는데, 모델이 업데이트되면 이 가이드도 다시 만들어야 할까요? 모델 버전에 따른 파라미터 민감도를 어떻게 관리할 수 있을까요?
2. 용도에 따라 최적 파라미터가 다른 것은, 결국 "정확성 vs 다양성" 트레이드오프입니다. 이 둘이 동시에 중요한 용도(예: 교육용 설명 생성)에는 어떤 전략이 좋을까요?
3. 실제 서비스에서 생성 파라미터를 사용자에게 노출해야 할까요, 아니면 백엔드에서 용도별로 고정해야 할까요? 각각의 장단점을 비교해보세요.